<a href="https://colab.research.google.com/github/volynets-d-v-ip54mp/graphic_processing_lab4-/blob/main/labs/pr4_python_numba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/YKochura/ac-kpi/blob/main/labs/pr4_python_numba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Прискорення обчислень нейронної мережі на GPU

Практична робота з перенесення обчислень прихованого шару нейронної мережі з CPU на GPU за допомогою бібліотеки Numba.


##  Імпорти та ініціалізація початкових значень для обчислень

In [2]:
import numpy as np
from numba import cuda, vectorize

# Прихований шар міститиме 100M нейронів

n1m = 1_000_000
n100m = 100_000_000

def generate_data(n):
    greyscales = np.floor(np.random.uniform(0, 255, n).astype(np.float32))
    weights = np.random.normal(.5, .1, n).astype(np.float32)
    return greyscales, weights

grayscales, weights = generate_data(n100m)

In [3]:
# TODO 1: numpy.exp не працює на GPU — замініть цей імпорт
from numba import cuda, vectorize

# TODO 2: додайте декоратор для запуску на GPU
def normalize(grayscales):
    return grayscales / 255

def weigh(values, weights):
    return values * weights

def activate(values):
    return ( np.exp(values) - np.exp(-values) ) / ( np.exp(values) + np.exp(-values) )

# TODO 3: оптимізуйте передачу даних — перенесіть масиви на GPU
# один раз, виконайте всі операції там, поверніть результат на CPU
def create_hidden_layer(greyscales, weights):
    normalized = normalize(greyscales)
    weighted   = weigh(normalized, weights)
    activated  = activate(weighted)
    # activated має бути масивом хоста (CPU)
    return activated

_ = create_hidden_layer(grayscales[:100], weights[:100])

Numba CPU

In [4]:
import math
@vectorize(['float32(float32)'], target='cpu')
def normalize_cpu(grayscales):
    return grayscales / 255

@vectorize(['float32(float32, float32)'], target='cpu')
def weigh_cpu(values, weights):
    return values * weights

@vectorize(['float32(float32)'], target='cpu')
def activate_cpu(values):
    return (math.exp(values) - math.exp(-values)) / (math.exp(values) + math.exp(-values))

def create_hidden_layer_cpu(greyscales, weights):
    normalized = normalize_cpu(greyscales)
    weighted   = weigh_cpu(normalized, weights)
    activated  = activate_cpu(weighted)
    return activated

# Прогрівання
_ = create_hidden_layer_cpu(grayscales[:100], weights[:100])

Numba GPU

In [5]:
@vectorize(['float32(float32)'], target='cuda')
def normalize_gpu(grayscales):
    return grayscales / 255.0

@vectorize(['float32(float32, float32)'], target='cuda')
def weigh_gpu(values, weights):
    return values * weights

@vectorize(['float32(float32)'], target='cuda')
def activate_gpu(values):
    return (math.exp(values) - math.exp(-values)) / (math.exp(values) + math.exp(-values))

# TODO 3: оптимізація передачі даних
def create_hidden_layer_gpu(greyscales, weights):
    # Відправляємо масиви на GPU
    d_greyscales = cuda.to_device(greyscales)
    d_weights = cuda.to_device(weights)

    # Виконуємо операції. Numba знає, що дані вже на GPU,
    # тому і результати (d_normalized, d_weighted, d_activated) створюються теж на GPU
    d_normalized = normalize_gpu(d_greyscales)
    d_weighted = weigh_gpu(d_normalized, d_weights)
    d_activated = activate_gpu(d_weighted)

    # Повертаємо фінальний результат на хост
    return d_activated.copy_to_host()

# Прогрівання JIT (компіляція CUDA ядер)
_ = create_hidden_layer_gpu(grayscales[:100], weights[:100])

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [6]:
print("Звичайний NumPy: ")
%timeit create_hidden_layer(grayscales, weights)

print("\nNumba CPU: ")
%timeit create_hidden_layer_cpu(grayscales, weights)

print("\nNumba GPU: ")
%timeit create_hidden_layer_gpu(grayscales, weights)

Звичайний NumPy: 
1.45 s ± 59.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Numba CPU: 
1.71 s ± 336 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Numba GPU: 
377 ms ± 26.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
greyscales_1m, weights_1m = generate_data(n1m)

print("Звичайний NumPy з n = 1m: ")
%timeit create_hidden_layer(greyscales_1m, weights_1m)

print("\nNumba CPU з n = 1m: ")
%timeit create_hidden_layer_cpu(greyscales_1m, weights_1m)

print("\nNumba GPU з n = 1m: ")
%timeit create_hidden_layer_gpu(greyscales_1m, weights_1m)

Звичайний NumPy з n = 1m: 
14.3 ms ± 2.1 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)

Numba CPU з n = 1m: 
14.8 ms ± 2.55 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)

Numba GPU з n = 1m: 
6.08 ms ± 446 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
